# Fire Detection Dataset Preprocessing

This notebook extracts the dataset archive, resizes all images to 224x224, and saves a balanced train/val/test dataset for model training.


In [2]:
from pathlib import Path
import zipfile

from PIL import Image
import numpy as np

ROOT = Path.cwd()
ARCHIVE_PATH = ROOT / 'data'
EXTRACT_DIR = ROOT / 'dataset'
PROCESSED_DIR = ROOT / 'processed'
IMAGE_SIZE = (224, 224)
LABELS = ['fire', 'nofire', 'smoke', 'smokefire']
SPLITS = ['train', 'val', 'test']


def extract_dataset():
    if not ARCHIVE_PATH.exists():
        raise FileNotFoundError(f'Dataset archive not found at {ARCHIVE_PATH}')

    if not EXTRACT_DIR.exists() or not any(EXTRACT_DIR.iterdir()):
        with zipfile.ZipFile(ARCHIVE_PATH, 'r') as archive:
            archive.extractall(EXTRACT_DIR)

    dataset_root = EXTRACT_DIR / 'Forect Fire' / 'Forest Fire_Dataset'
    if not dataset_root.exists():
        raise FileNotFoundError(f'Expected extracted dataset folder at {dataset_root}')
    return dataset_root


def preprocess_image(src_path: Path, dst_path: Path):
    with Image.open(src_path) as img:
        rgb = img.convert('RGB')
        resized = rgb.resize(IMAGE_SIZE)
        arr = np.asarray(resized, dtype=np.float32) / 255.0
        arr = (arr * 255.0).astype('uint8')
        Image.fromarray(arr, mode='RGB').save(dst_path)


def main():
    dataset_root = extract_dataset()
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

    for split in SPLITS:
        split_dir = dataset_root / split
        if not split_dir.exists():
            continue

        for label in LABELS:
            src_dir = split_dir / label
            dst_dir = PROCESSED_DIR / split / label
            dst_dir.mkdir(parents=True, exist_ok=True)

            if not src_dir.exists():
                continue

            for image_path in sorted(src_dir.glob('*.jpg')):
                out_path = dst_dir / image_path.name
                preprocess_image(image_path, out_path)

    print(f'Processed dataset saved to {PROCESSED_DIR}')
